# Sesión 8 · Parte 2 de 3 — APIs y autenticación

Este notebook es autoexplicativo: cada tema incluye su **definición**, la **sintaxis** que se usa,
**para qué sirve** cada elemento, y un **ejemplo ejecutable**.

Segunda de tres partes de la Sesión 8. La Parte 1 (Bases de datos y repositorios) cubrió cómo leer datos
ya publicados; aquí se ve cómo un programa **solicita** datos directamente a un servicio, incluyendo un
caso real con autenticación: la API de Kaggle.

### ¿Qué es una API?

> **Definición:** Una **API** (*Application Programming Interface*) es un punto de acceso que un servicio
> pone a disposición para que un programa —no una persona— solicite datos o acciones, sin necesitar saber
> cómo funciona el servicio por dentro.

**Para qué sirve:** en vez de que cada aplicación tenga que replicar la información de un servicio, todas
consultan el mismo punto de acceso y reciben los datos en un formato estándar.

El flujo siempre es el mismo: tu programa (**cliente**) envía una **solicitud** (*request*) a un
**servidor**, y el servidor regresa una **respuesta** (*response*).

### El endpoint y la librería `requests`

> **Definición:** Un **endpoint** es la URL específica del recurso que se está pidiendo. El **método**
> más común para leer datos es `GET`.

**Sintaxis:**

```python
import requests

respuesta = requests.get(endpoint)
```

**Para qué sirve:** `requests.get()` envía la solicitud y regresa un objeto de respuesta con todo lo que
el servidor contestó: el contenido, el código de estado, los encabezados, etc.

PyPI (el repositorio oficial de paquetes de Python) ofrece una API pública: por cada paquete existe un
endpoint que regresa su información en formato JSON, sin necesitar autenticación.

**Antes de ejecutar la siguiente celda, escribe tu predicción:**

¿Qué tipo de dato de Python esperas recibir cuando se llama a `.json()` sobre la respuesta?

In [1]:
import requests

endpoint = "https://pypi.org/pypi/pandas/json"
respuesta = requests.get(endpoint)

print(respuesta.status_code)

200


### `status_code`

> **Definición:** `status_code` es el número que indica si la solicitud tuvo éxito. Es parte del
> protocolo HTTP, el mismo que usa cualquier navegador.

**Sintaxis:** `respuesta.status_code`

**Para qué sirve:** revisarlo *antes* de usar los datos evita errores silenciosos. Los rangos más comunes:

| Código | Significado |
|---|---|
| 200 | Éxito |
| 400 | Solicitud mal formada (error del cliente) |
| 401 / 403 | No autorizado / prohibido (falta autenticación o no se tiene permiso) |
| 404 | El recurso no existe |
| 429 | Se excedió el límite de peticiones permitidas (*rate limit*) |
| 500 | Error del servidor |

Con la petición confirmada, se convierte la respuesta a una estructura de Python.

In [2]:
datos = respuesta.json()
print(type(datos))
print(list(datos.keys()))

<class 'dict'>
['info', 'last_serial', 'ownership', 'releases', 'urls', 'vulnerabilities']


### JSON como dict/list

> **Definición:** **JSON** (*JavaScript Object Notation*) es el formato estándar para intercambiar datos
> entre sistemas. Un objeto JSON `{ }` es equivalente a un `dict` de Python; un arreglo JSON `[ ]` es
> equivalente a una `list`. Esto no es nuevo: es la misma estructura anidada que se trabajó desde Sesión 1.

**Sintaxis:** una vez que `datos` es un `dict`, se navega igual que cualquier diccionario anidado:
`datos["llave"]["sub_llave"]`.

**Para qué sirve:** casi ninguna API entrega datos planos — la información relevante casi siempre está
anidada dentro de una o más llaves, y hay que saber entrar a buscarla.

In [3]:
info = datos["info"]
print(info["name"])
print(info["version"])
print(info["summary"])

pandas
3.0.5
Powerful data structures for data analysis, time series, and statistics


**Antes de ejecutar la siguiente celda, escribe tu predicción:**

¿Qué error esperas si se intenta acceder a `datos["version"]` en vez de `datos["info"]["version"]`?

In [4]:
# Esta celda produce un error intencionalmente
datos["version"]

KeyError: 'version'

El error es `KeyError`: la llave `"version"` no existe en el nivel superior del diccionario, sólo
dentro de `"info"`. Cuando una API regresa datos anidados, hay que navegar la estructura nivel por nivel.

### Parámetros

> **Definición:** Los **parámetros** son filtros o instrucciones adicionales que se añaden a la solicitud,
> típicamente para pedir un subconjunto específico de datos en vez de todo.

**Sintaxis:**

```python
requests.get(endpoint, params={"clave": "valor"})
```

**Para qué sirve:** evita descargar más información de la necesaria. `requests` arma la URL final
automáticamente a partir del diccionario — no hay que construir el string `?clave=valor` a mano.

In [5]:
# Ejemplo con parámetros: buscar paquetes en PyPI que contengan una palabra
endpoint_busqueda = "https://pypi.org/search/"
respuesta_busqueda = requests.get(endpoint_busqueda, params={"q": "pandas"})

print(respuesta_busqueda.status_code)
print(respuesta_busqueda.url)

200
https://pypi.org/search/?q=pandas


### Ejemplo integrador: pronóstico del clima con Open-Meteo

Hasta ahora cada ejemplo aisló un concepto. Este junta todo lo visto —endpoint, parámetros, `status_code`,
JSON— en un caso real: obtener el pronóstico del clima para Ciudad Juárez desde **Open-Meteo**, una API
pública que no requiere autenticación.

> **Definición:** `timeout` es el número máximo de segundos que se espera una respuesta antes de darse
> por vencido. Sin él, una solicitud que nunca responde puede dejar el programa congelado indefinidamente.

> **Definición:** `respuesta.raise_for_status()` revisa el `status_code` y **lanza un error** si no fue
> exitoso (400 o más), en vez de tener que escribir un `if` manualmente cada vez.

**Sintaxis para manejar errores de red:**

```python
try:
    respuesta = requests.get(endpoint, params=parametros, timeout=30)
    respuesta.raise_for_status()
    datos = respuesta.json()
except requests.RequestException as error:
    print("No fue posible consultar la API:", error)
```

**Para qué sirve el `try/except` aquí:** una solicitud de red puede fallar por muchas razones ajenas al
código (sin internet, el servidor no responde, tiempo agotado). Sin este bloque, cualquiera de esas fallas
detendría el notebook con un error. Con él, el programa puede reaccionar de forma controlada.

In [6]:
import requests
import pandas as pd

endpoint = "https://api.open-meteo.com/v1/forecast"

# latitude/longitude ubican Ciudad Juárez; hourly especifica qué variables se piden por hora
parametros = {
    "latitude": 31.69,
    "longitude": -106.42,
    "hourly": "temperature_2m,relative_humidity_2m,precipitation_probability",
    "forecast_days": 2,
    "timezone": "auto",
}

try:
    respuesta = requests.get(endpoint, params=parametros, timeout=30)
    respuesta.raise_for_status()
    datos = respuesta.json()
    origen_respuesta = "Open-Meteo (consulta en línea)"
    print("status_code:", respuesta.status_code)

except requests.RequestException as error:
    # Si no hay conexión a Open-Meteo (por ejemplo, en un entorno con acceso a internet restringido),
    # se usa una respuesta de respaldo con la misma estructura, para poder seguir la práctica.
    print("No fue posible consultar la API:", error)
    print("Se usará una respuesta de respaldo, claramente marcada como tal.")
    origen_respuesta = "Respuesta de respaldo (no es una consulta en línea)"
    datos = {
        "latitude": 31.69,
        "longitude": -106.42,
        "timezone": "America/Ciudad_Juarez",
        "hourly_units": {
            "time": "iso8601",
            "temperature_2m": "°C",
            "relative_humidity_2m": "%",
            "precipitation_probability": "%",
        },
        "hourly": {
            "time": ["2026-08-29T00:00", "2026-08-29T01:00", "2026-08-29T02:00"],
            "temperature_2m": [27.4, 26.8, 26.1],
            "relative_humidity_2m": [36, 38, 40],
            "precipitation_probability": [10, 12, 15],
        },
    }

print("Origen utilizado:", origen_respuesta)

status_code: 200
Origen utilizado: Open-Meteo (consulta en línea)


### JSON, con más detalle

La tabla de equivalencia completa entre JSON y Python:

| JSON | Python |
|---|---|
| objeto `{ }` | `dict` |
| arreglo `[ ]` | `list` |
| cadena | `str` |
| número | `int` o `float` |
| verdadero / falso | `True` / `False` |
| nulo | `None` |

La respuesta de Open-Meteo trae metadatos, unidades, y los registros por hora — no todo tiene la misma
forma, así que conviene explorar antes de asumir la estructura.

In [7]:
print("Tipo del objeto principal:", type(datos))
print("Claves principales:", list(datos.keys()))
print("Claves de 'hourly':", list(datos["hourly"].keys()))
print("Unidades:", datos.get("hourly_units", {}))

Tipo del objeto principal: <class 'dict'>
Claves principales: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
Claves de 'hourly': ['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation_probability']
Unidades: {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'precipitation_probability': '%'}


**Antes de ejecutar la siguiente celda, escribe tu predicción:**

Dentro de `datos["hourly"]`, cada llave contiene una lista. ¿Qué relación debe existir entre las
longitudes de esas listas para poder construir un `DataFrame` correctamente?

In [8]:
longitudes = {
    variable: len(valores)
    for variable, valores in datos["hourly"].items()
}

longitudes

{'time': 48,
 'temperature_2m': 48,
 'relative_humidity_2m': 48,
 'precipitation_probability': 48}

### Construir un DataFrame desde un diccionario de listas

> **Sintaxis:** `pd.DataFrame(diccionario_de_listas)` — cada llave se convierte en una columna, y cada
> posición dentro de las listas se convierte en una fila.

**Para qué sirve:** es el paso final de la adquisición por API — pasar de la estructura JSON anidada a un
`DataFrame` listo para analizar, igual que con `read_csv()` o `read_sql()`.

In [9]:
df_clima = pd.DataFrame(datos["hourly"])
df_clima["time"] = pd.to_datetime(df_clima["time"])

df_clima.head()

,time,temperature_2m,relative_humidity_2m,precipitation_probability
0,2026-08-28 00:00:00,25.8,47,2
1,2026-08-28 01:00:00,25.7,47,2
2,2026-08-28 02:00:00,25.5,47,1
3,2026-08-28 03:00:00,25.5,42,1
4,2026-08-28 04:00:00,25.3,45,1


Aplicando el mismo criterio de verificación ya usado con bases de datos y repositorios:

In [10]:
print("Dimensiones:", df_clima.shape)
print()
df_clima.info()
print()
print("Valores faltantes por columna:")
print(df_clima.isna().sum())

Dimensiones: (48, 4)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   time                       48 non-null     datetime64[ns]
 1   temperature_2m             48 non-null     float64       
 2   relative_humidity_2m       48 non-null     int64         
 3   precipitation_probability  48 non-null     int64         
dtypes: datetime64[ns](1), float64(1), int64(2)
memory usage: 1.6 KB

Valores faltantes por columna:
time                         0
temperature_2m               0
relative_humidity_2m         0
precipitation_probability    0
dtype: int64


### Ficha de procedencia

Al igual que con un dataset descargado, una adquisición por API se documenta: de dónde vino, con qué
parámetros, y cuándo se consultó.

In [11]:
from datetime import datetime, timezone

ficha_api = {
    "servicio": "Open-Meteo",
    "endpoint": endpoint,
    "parametros": parametros,
    "fecha_consulta_utc": datetime.now(timezone.utc).isoformat(),
    "unidades": datos.get("hourly_units", {}),
    "origen_respuesta": origen_respuesta,
}

ficha_api

{'servicio': 'Open-Meteo',
 'endpoint': 'https://api.open-meteo.com/v1/forecast',
 'parametros': {'latitude': 31.69,
  'longitude': -106.42,
  'hourly': 'temperature_2m,relative_humidity_2m,precipitation_probability',
  'forecast_days': 2,
  'timezone': 'auto'},
 'fecha_consulta_utc': '2026-08-29T03:32:57.234498+00:00',
 'unidades': {'time': 'iso8601',
  'temperature_2m': '°C',
  'relative_humidity_2m': '%',
  'precipitation_probability': '%'},
 'origen_respuesta': 'Open-Meteo (consulta en línea)'}

### Errores y límites que conviene anticipar

No toda API se comporta igual que PyPI u Open-Meteo. Antes de trabajar con una nueva, conviene tener en
mente:

- **Parámetro incorrecto** — la API puede rechazar la solicitud con 400.
- **Tiempo de espera** — el servidor no responde dentro del `timeout`.
- **Autenticación** — algunas APIs requieren token o key (como se vio arriba con Kaggle).
- **Cuota** — puede existir un máximo de solicitudes permitidas (rate limit, código 429).
- **Paginación** — una respuesta puede traer sólo una parte de los registros totales.
- **Cambio de versión** — el endpoint o los campos que regresa pueden modificarse con el tiempo.
- **Respuesta válida pero inesperada** — un `status_code` 200 no garantiza que los datos sean los que se
  esperaban; siempre conviene verificar la estructura antes de confiar en ella.

### Autenticación

> **Definición:** La **autenticación** es el mecanismo por el cual una API identifica quién hace la
> solicitud. Las dos formas más comunes son la **API key** (una cadena única asociada a una cuenta) y el
> **token** (una credencial temporal, a veces obtenida tras iniciar sesión).

**Sintaxis típica:** la credencial se envía en los **encabezados** (*headers*) de la solicitud, no en la URL:

```python
headers = {"Authorization": "Bearer TU_API_KEY"}
requests.get(endpoint, headers=headers)
```

**Para qué sirve:** permite que el servicio sepa quién hace cada solicitud, aplique límites de uso por
cuenta, y restrinja el acceso a datos privados.

**Regla de seguridad, sin excepción:** una API key nunca se escribe directamente en el código de un
notebook que se comparte o se sube a GitHub. Se guarda por fuera del código y se carga en tiempo de
ejecución — más abajo en este mismo notebook se aplica con un caso real: la API de Kaggle.

No todas las solicitudes tienen éxito. Es buena práctica revisar siempre `status_code` antes de continuar.

In [12]:
endpoint_inexistente = "https://pypi.org/pypi/paquete-que-no-existe-xyz123/json"
respuesta_fallida = requests.get(endpoint_inexistente)

print(respuesta_fallida.status_code)

if respuesta_fallida.status_code == 200:
    print("Solicitud exitosa, se puede continuar")
else:
    print("La solicitud no tuvo éxito, no se debe intentar leer .json() con confianza")

404
La solicitud no tuvo éxito, no se debe intentar leer .json() con confianza


### Descarga desde Kaggle con autenticación

> **Definición:** Kaggle ofrece una **API propia** para descargar datasets por código, en vez de hacerlo
> manualmente desde el navegador. Requiere autenticación mediante un **token**: una cadena única generada
> desde tu cuenta de Kaggle.

**Cómo obtener tu token de Kaggle, paso a paso:**

1. Entra a [kaggle.com](https://kaggle.com) e inicia sesión (o crea una cuenta gratuita si no tienes una).
2. Haz clic en tu foto de perfil, en la esquina superior derecha, y entra a **Settings**.
3. Baja hasta la sección **API** y haz clic en **Create New Token**.
4. Aparece una ventana titulada **"API Token is now available"** con tu token (empieza con `KGAT_...`). **Cópialo en ese momento** — Kaggle lo muestra una sola vez y no lo podrás volver a ver (si lo pierdes, generas uno nuevo repitiendo este paso).
5. En Google Colab, haz clic en el ícono de llave 🔑 en el panel izquierdo (**Secretos**).
6. Crea un secreto llamado exactamente `KAGGLE_API_TOKEN`, y en `value`pega el valor del token que copiaste.
7. Activa el interruptor de acceso de ese secreto para que **este notebook** pueda leerlo — sin eso, Colab lo guarda pero no se lo entrega a las celdas.

**Por qué no se pega el token directamente en el código:** siguiendo la regla de seguridad de arriba, el
token se guarda en los Secretos de Colab en vez de escribirse en una celda — nunca queda visible en el
código ni se sube a GitHub por accidente.

**Sintaxis:**

```python
from google.colab import userdata
import os

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
```

**Para qué sirve:** una vez configurada esa variable de entorno, la librería `kaggle` se autentica
automáticamente en cada solicitud, sin que el token aparezca en ninguna celda visible.

In [16]:
!pip install kaggle --quiet

import os

# Esta celda sólo funciona dentro de Google Colab, con el Secreto configurado.
# Fuera de Colab se omite con un aviso.
try:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    print("Token de Kaggle cargado desde Secretos de Colab")
except ImportError:
    print("Esta celda está diseñada para ejecutarse en Google Colab con el Secreto configurado")

Token de Kaggle cargado desde Secretos de Colab


**Antes de ejecutar la siguiente celda, escribe tu predicción:**

Si `KAGGLE_API_TOKEN` no está configurado en los Secretos de Colab, ¿qué esperas que pase al intentar
descargar el dataset?

El dataset que se descarga en este notebook es **`uciml/iris`** (el dataset clásico de flores iris). Si tu
profesor indica un dataset distinto, sólo cambia el identificador.

In [17]:
dataset_id = "uciml/iris"  # Aquí puedes cambiar si requieres otro conjunto de datos

!kaggle datasets download -d {dataset_id} -p datos_kaggle/ --unzip

Dataset URL: https://www.kaggle.com/datasets/uciml/iris
License(s): CC0-1.0
100% 3.60k/3.60k [00:00<00:00, 9.81MB/s]



**Para qué sirve `-p datos_kaggle/`:** indica en qué carpeta se guarda el archivo descargado.
**Para qué sirve `--unzip`:** Kaggle entrega los datasets comprimidos; esta bandera los descomprime
automáticamente, sin necesitar un paso aparte.

Una vez descargado, se lee exactamente igual que cualquier otro `.csv` — el mismo `read_csv()` visto en
la Parte 1, y el mismo criterio de documentar la procedencia (diccionario de datos) explicado ahí.

In [18]:
import glob

archivos_csv = glob.glob("datos_kaggle/*.csv")
print(archivos_csv)

if archivos_csv:
    df_kaggle = pd.read_csv(archivos_csv[0])
    print(df_kaggle.shape)
    df_kaggle.head()

['datos_kaggle/Iris.csv']
(150, 6)


### Conexión con UCI Machine Learning Repository

> **Definición:** UCI ofrece su propio paquete de Python, `ucimlrepo`, que descarga un dataset directamente
> a partir de su identificador — sin necesitar autenticación, a diferencia de Kaggle.

**Cómo encontrar el identificador de un dataset:** en la página del dataset dentro de
[archive.ics.uci.edu](https://archive.ics.uci.edu/), el identificador aparece en la URL, o se puede buscar
por nombre con `list_available_datasets()`.

**Sintaxis:**

```python
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=45)
X = dataset.data.features   # variables predictoras, como DataFrame
y = dataset.data.targets    # variable objetivo, como DataFrame
```

**Para qué sirve cada parte:** `fetch_ucirepo()` regresa un objeto con tres partes: `.data` (los datos ya
separados en `features` y `targets`), `.metadata` (información general del dataset) y `.variables` (una
tabla que describe cada columna — el equivalente al diccionario de datos visto en la Parte 1).

In [19]:
!pip install ucimlrepo --quiet

from ucimlrepo import fetch_ucirepo

# id=45 corresponde al dataset "Heart Disease", uno de los más usados del repositorio
try:
    dataset_uci = fetch_ucirepo(id=45)
    X = dataset_uci.data.features
    y = dataset_uci.data.targets
    origen_uci = "UCI ML Repository (consulta en línea)"

except Exception as error:
    # Si no hay conexión al repositorio (por ejemplo, en un entorno con acceso a internet restringido),
    # se usa una tabla de respaldo pequeña, con la misma forma, para poder seguir la práctica.
    print("No fue posible consultar UCI ML Repository:", error)
    print("Se usará una tabla de respaldo, claramente marcada como tal.")
    origen_uci = "Tabla de respaldo (no es una consulta en línea)"
    X = pd.DataFrame({
        "edad": [63, 37, 41, 56],
        "colesterol": [233, 250, 204, 236],
    })
    y = pd.DataFrame({"diagnostico": [0, 1, 0, 1]})

print("Origen utilizado:", origen_uci)
print("Dimensiones de X:", X.shape)
X.head()

Origen utilizado: UCI ML Repository (consulta en línea)
Dimensiones de X: (303, 13)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0


`y` contiene la variable objetivo por separado — útil más adelante, cuando el curso llegue a modelos
predictivos (Tema 4). Por ahora basta con reconocer que un dataset de UCI puede llegar ya dividido en
variables predictoras y variable objetivo, distinto al `DataFrame` único que entrega `read_csv()`.

## Fuera de alcance de esta parte

- Autenticación mediante OAuth (login de usuario dentro de la aplicación)
- Paginación de resultados en APIs
- Manejo avanzado de límites de tasa (reintentos, backoff)

La autenticación con API key **sí** está en el alcance: se usó arriba con la API de Kaggle.

## Glosario de esta parte

| Término | Significado |
|---|---|
| API | Punto de acceso que un servicio ofrece para que un programa solicite datos |
| Endpoint | Dirección específica de la API que se consulta |
| Request / Response | Solicitud enviada al servidor / respuesta recibida |
| `status_code` | Número que indica si la solicitud tuvo éxito (200 = éxito) |
| JSON | Formato estándar de intercambio de datos, equivalente a dict/list en Python |
| Parámetros | Filtros añadidos a una solicitud para pedir un subconjunto de datos |
| `timeout` | Tiempo máximo de espera antes de darse por vencido con una solicitud |
| `raise_for_status()` | Lanza un error automáticamente si el status_code no fue exitoso |
| Autenticación | Mecanismo para identificar quién hace la solicitud |
| API key / token | Credencial que identifica a quien hace la solicitud |
| Encabezados (headers) | Metadatos de la solicitud, incluida la credencial de autenticación |
| Rate limit | Límite de solicitudes permitidas en un periodo de tiempo |
| Secretos de Colab | Gestor donde se guardan credenciales sin escribirlas en el código |
| Ficha de procedencia | Registro de qué se consultó, con qué parámetros y cuándo |